# San Francisco income and bus service

In [ ]:
import json
from pathlib import Path

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from IPython.display import display

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / 'data/cleaned/notebook'
VIS_DIR = PROJECT_DIR / 'outputs/visualizations'
ARRIVAL_DIR = PROJECT_DIR / 'outputs/arrival_divergence'
SERVICE_DIR = PROJECT_DIR / 'outputs/income_access_service'
for folder in [VIS_DIR, ARRIVAL_DIR, SERVICE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

metadata = json.loads((DATA_DIR / 'analysis_metadata.json').read_text())

## Income data

The builder combines ACS 2024 estimates, ACS income brackets by year, and IRS 2020–2022 ZIP totals. ZIP 94104 has no published ACS median household income.

In [ ]:
income = pd.read_csv(DATA_DIR / 'sf_zip_socioeconomic_table.csv', dtype={'zip': str})
availability = pd.read_csv(DATA_DIR / 'income_availability.csv')
display(income.head())
display(availability)

## Income brackets and ZIP outliers

Orange points mark ZIPs outside the 1.5 × IQR whiskers. Only brackets with outliers are shown.

In [ ]:
box_shares = pd.read_csv(DATA_DIR / 'income_box_shares.csv')
outliers = pd.read_csv(DATA_DIR / 'income_bracket_outliers.csv', dtype={'zip': str})

fig, ax = plt.subplots(figsize=(12, 9))
ax.boxplot(
    box_shares, orientation='horizontal', showfliers=False, patch_artist=True,
    boxprops={'facecolor': 'lightblue', 'edgecolor': 'darkslategray'},
    medianprops={'color': 'midnightblue', 'linewidth': 2}
)
for _, row in outliers.iterrows():
    ax.scatter(row['share_pct'], row['y_position'], color='darkorange')
    ax.annotate(
        row['zip'], (row['share_pct'], row['y_position']),
        xytext=(5, 0), textcoords='offset points', va='center', fontsize=8
    )
ax.set_yticks(range(1, len(box_shares.columns) + 1), box_shares.columns)
ax.invert_yaxis()
ax.set_xlim(left=0)
ax.set_xlabel('Share of households in this income bracket (%)')
ax.set_ylabel('Household income bracket')
ax.set_title('2024 ACS income-bracket shares with ZIP outliers')
ax.scatter([], [], color='darkorange', label='ZIP outside 1.5 × IQR whiskers')
ax.legend()
ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
fig.savefig(VIS_DIR / 'sf_zip_acs_bracket_share_outliers.png', dpi=180, bbox_inches='tight')
plt.show()

## Income groups and transit stops

The prepared score ranks ZIPs using the household shares below $25,000 and above $100,000, then divides the ZIPs into three groups.

In [ ]:
income_map = gpd.read_parquet(DATA_DIR / 'income_zip_map.parquet')
streets = gpd.read_parquet(DATA_DIR / 'sf_street_edges.parquet')
stops = gpd.read_parquet(DATA_DIR / 'sf_transit_stops.parquet')
legend = pd.read_csv(DATA_DIR / 'income_map_legend.csv')
bounds = pd.read_csv(DATA_DIR / 'income_map_bounds.csv').iloc[0]

fig, ax = plt.subplots(figsize=(10, 8))
income_map.plot(ax=ax, color=income_map['map_color'], alpha=0.24, edgecolor='none')
streets.plot(ax=ax, color='darkgray', linewidth=0.12, alpha=0.48)
stops.plot(ax=ax, color='black', markersize=1.4, alpha=0.75, zorder=3)
income_map.boundary.plot(ax=ax, color=income_map['map_color'], linewidth=2)
legend_items = []
for _, row in legend.iterrows():
    legend_items.append(Patch(color=row['color'], alpha=0.7, label=row['label']))
legend_items.append(Line2D(
    [], [], marker='o', linestyle='none', color='black',
    markersize=3, label='Physical transit stop'
))
ax.legend(
    handles=legend_items, title='Income groups (income in thousands of dollars)',
    loc='upper left', fontsize=7, title_fontsize=8
)
ax.set_title('SF ZIP income groups and transit stops (ACS 2024)')
ax.set_xlim(bounds['xmin'], bounds['xmax'])
ax.set_ylim(bounds['ymin'], bounds['ymax'])
ax.set_axis_off()
fig.tight_layout()
fig.savefig(VIS_DIR / 'sf_zip_socioeconomic_status_street_network_map.png', dpi=200, bbox_inches='tight')
plt.show()

## Observed and scheduled arrival shares

Each share is the fraction of all SF arrivals occurring in one ZIP during a month. KL compares a ZIP with the rest of SF, and KS measures the largest gap between the two cumulative distributions.

These comparisons keep quality-flagged months visible. The 27 KS tests use paired months that may be related over time, so their p-values are descriptive.

In [ ]:
zip_results = pd.read_csv(DATA_DIR / 'per_zip_kl_ks_results.csv', dtype={'zip_code': str})
cdf_data = pd.read_csv(DATA_DIR / 'monthly_share_cdfs.csv', dtype={'zip_code': str})
plot_summary = pd.read_csv(DATA_DIR / 'arrival_plot_summary.csv', dtype={'zip_code': str})
print('Months:', metadata['month_count'])
print('Months with data quality flags:', metadata['quality_flagged_months'])
display(zip_results)

### ZIPs with KS p < 0.05

The orange segment marks the largest gap between the observed and scheduled distributions.

In [ ]:
flagged_zips = metadata['flagged_zip_codes']
plot_rows = max(1, (len(flagged_zips) + 1) // 2)
fig, axes = plt.subplots(plot_rows, 2, figsize=(12, 4 * plot_rows), squeeze=False)
axes = axes.flatten()

for i, zip_code in enumerate(flagged_zips):
    ax = axes[i]
    cdf = cdf_data.loc[cdf_data['zip_code'] == zip_code]
    result = plot_summary.loc[plot_summary['zip_code'] == zip_code].iloc[0]
    ax.step(cdf['share_pct'], cdf['observed'], where='post', label='Observed 511')
    ax.step(cdf['share_pct'], cdf['scheduled'], where='post', label='Scheduled SFMTA')
    ax.plot(
        [result['gap_share_pct'], result['gap_share_pct']],
        [result['observed_cdf_at_gap'], result['scheduled_cdf_at_gap']],
        color='darkorange', linewidth=3
    )
    ax.set_title(
        f'{zip_code} | KL={result["mean_monthly_kl"]:.6f}, '
        f'KS D={result["ks_statistic"]:.2f}, p={result["ks_p_value"]:.3g}'
    )
    ax.set_xlabel('ZIP share of SF arrivals in a month (%)')
    ax.set_ylabel('Fraction of months at or below share')
    ax.grid(alpha=0.2)
for i in range(len(flagged_zips), len(axes)):
    axes[i].set_visible(False)
axes[0].legend()
fig.suptitle('Observed and scheduled monthly shares for ZIPs with KS p < 0.05')
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(ARRIVAL_DIR / 'per_zip_flagged_cdfs.png', dpi=180, bbox_inches='tight')
plt.show()

### ZIP with the lowest KS p-value

When ZIPs tie, show the one with the larger mean KL value.

In [ ]:
selected_zip = metadata['most_significant_zip']
cdf = cdf_data.loc[cdf_data['zip_code'] == selected_zip]
result = plot_summary.loc[plot_summary['zip_code'] == selected_zip].iloc[0]
print('ZIPs tied at the lowest p-value:', metadata['tied_lowest_p_value_zips'])

fig, ax = plt.subplots(figsize=(9, 6))
ax.step(cdf['share_pct'], cdf['observed'], where='post', linewidth=2, label='Observed 511')
ax.step(cdf['share_pct'], cdf['scheduled'], where='post', linewidth=2, label='Scheduled SFMTA')
ax.plot(
    [result['gap_share_pct'], result['gap_share_pct']],
    [result['observed_cdf_at_gap'], result['scheduled_cdf_at_gap']],
    color='darkorange', linewidth=3,
    label=f'Maximum gap D = {result["ks_statistic"]:.3f}'
)
ax.set_title(
    f'ZIP {selected_zip}: observed and scheduled monthly shares\n'
    f'KS p = {result["ks_p_value"]:.6g}; KL = {result["mean_monthly_kl"]:.6f}'
)
ax.set_xlabel('ZIP share of SF arrivals in a month (%)')
ax.set_ylabel('Fraction of months at or below share')
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
fig.savefig(ARRIVAL_DIR / 'most_significant_zip_ks_cdf.png', dpi=180, bbox_inches='tight')
plt.show()

### Average difference in arrival shares

In [ ]:
share_gaps = pd.read_csv(DATA_DIR / 'arrival_share_gaps.csv', dtype={'zip_code': str})
fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(share_gaps['zip_code'], share_gaps['mean_share_gap_pp'], color=share_gaps['color'])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean observed minus scheduled share (percentage points)')
ax.set_ylabel('ZIP code')
ax.set_title('Average difference in ZIP share across 55 months')
ax.legend(handles=[
    Patch(color='darkorange', label='KS p < 0.05'),
    Patch(color='steelblue', label='KS p >= 0.05')
])
fig.tight_layout()
fig.savefig(ARRIVAL_DIR / 'per_zip_share_gaps.png', dpi=180, bbox_inches='tight')
plt.show()

## Access and service

The builder summarizes only months passing both data quality rules for these charts. Frequency and on-time performance use totals across those months.

The next two charts preserve the service comparisons contributed in the separate income and access notebook. ZIP 94104 is excluded because its median income is unavailable.

In [ ]:
income_service = pd.read_csv(DATA_DIR / 'income_service_charts.csv', dtype={'zip_code': str})
chart_labels = pd.read_csv(DATA_DIR / 'service_chart_labels.csv', dtype={'zip_code': str})
print('ZIPs in the income and service charts:', metadata['income_service_zip_count'])
display(income_service.head())

### Stops per resident and bus frequency

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    income_service['stops_per_1000_residents'],
    income_service['arrivals_per_stop_per_day'], color='steelblue'
)
for _, row in chart_labels.iterrows():
    ax.text(row['label_x'], row['label_y'], row['zip_code'], va='center')
ax.set_xlabel('Bus stops per 1,000 residents')
ax.set_ylabel('Bus arrivals per stop per day')
ax.set_title('SF bus stops per resident and bus frequency, 2022–2026')
fig.tight_layout()
fig.savefig(SERVICE_DIR / 'stops_per_resident_vs_frequency.png', bbox_inches='tight')
plt.show()
print('Spearman rho:', round(metadata['stops_frequency_spearman'], 2))

### On-time performance, ordered by income

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
ax.scatter(income_service['on_time_pct'], income_service['income_label'], color='steelblue')
ax.grid(axis='y', color='lightgray')
ax.set_axisbelow(True)
ax.set_title('SF on-time bus performance by ZIP, ordered by income, 2022–2026')
ax.set_xlabel('Bus arrivals on time (%)')
ax.set_ylabel('ZIP code (median household income, thousands of dollars)')
fig.tight_layout()
fig.savefig(SERVICE_DIR / 'on_time_by_income.png', bbox_inches='tight')
plt.show()
print('Income and on-time performance, Spearman rho:', round(metadata['income_ontime_spearman'], 2))

### ZIP summary and access maps

In [ ]:
service_summary = pd.read_csv(DATA_DIR / 'service_summary_statistics.csv')
service_map = gpd.read_parquet(DATA_DIR / 'service_zip_map.parquet')
display(service_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
service_map.plot(
    column='mean_stops_per_sqmi', cmap='Blues', legend=True,
    edgecolor='white', ax=axes[0]
)
axes[0].set_title('Average scheduled bus stops per square mile')
axes[0].set_axis_off()
service_map.plot(
    column='quarter_mile_stop_coverage_pct_2026', cmap='Greens', legend=True,
    edgecolor='white', ax=axes[1]
)
axes[1].set_title('Land within a quarter mile of a bus stop (2026)')
axes[1].set_axis_off()
fig.tight_layout()
fig.savefig(VIS_DIR / 'transit_access_maps.png', bbox_inches='tight')
plt.show()

### Monthly service trends

In [ ]:
city_service = pd.read_csv(DATA_DIR / 'monthly_service_summary.csv', parse_dates=['year_month'])
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(city_service['year_month'], city_service['arrivals_per_stop_per_day'], color='steelblue')
axes[0].set_ylabel('Arrivals per stop per day')
axes[0].set_title('Citywide Muni bus service in quality-passing months')
axes[1].plot(city_service['year_month'], city_service['on_time_pct'], color='navy')
axes[1].set_ylabel('On-time arrivals (%)')
axes[1].set_xlabel('Month')
fig.tight_layout()
fig.savefig(VIS_DIR / 'trend_citywide.png', bbox_inches='tight')
plt.show()

### Income and bus frequency

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    income_service['median_household_income'],
    income_service['arrivals_per_stop_per_day'], color='steelblue'
)
ax.set_xlabel('Median household income (US dollars)')
ax.set_ylabel('Bus arrivals per stop per day')
ax.set_title('Income and observed Muni bus frequency')
fig.tight_layout()
fig.savefig(VIS_DIR / 'income_and_service.png', bbox_inches='tight')
plt.show()

## Correlations and regression models

These results are calculated in the Python builder. The models compare bus frequency and on-time performance with access, income, and poverty. Each model uses one row per ZIP with complete values.

In [ ]:
correlations = pd.read_csv(DATA_DIR / 'service_correlations.csv')
model_coefficients = pd.read_csv(DATA_DIR / 'service_model_coefficients.csv')
display(correlations.round(3))
display(model_coefficients.round(3))